# Blindspot · SFT Training & Analysis (Colab + A100)

End-to-end SFT training notebook for the **Blindspot** OpenEnv.
Fine-tunes **Qwen2.5-1.5B-Instruct (4-bit LoRA via Unsloth)** on 40 expert
traces generated by the Dense Retrieval+ heuristic (+8.67 mean reward).

Sections:
1. **Setup** — install deps, clone repo, sanity checks.
2. **Pre-training analysis** — explore dataset distributions, run baselines.
3. **SFT Training** — load model, attach LoRA r=16, run SFTTrainer for 1 epoch.
4. **Post-training analysis** — evaluate SFT policy vs baselines, plot results.

Estimated cost on A100: ~15–25 minutes wall time, ~3–5 compute units.


---
## 1. Setup

In [ ]:
%%bash
# STEP 0 — run this first, every session: wipes any stale clone so later cells never conflict
rm -rf /content/blindspot-env
echo "clean"


In [ ]:
%%bash
set -e
# Install SFT dependencies: Unsloth (Qwen2.5 support), TRL SFTTrainer, datasets
pip install -q --upgrade pip
pip install -q --upgrade --no-cache-dir \
  "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
  "unsloth_zoo" \
  "trl>=0.17.0" \
  "datasets>=2.18.0" \
  "transformers>=4.40.0" \
  "accelerate" \
  "bitsandbytes" \
  "peft" \
  "huggingface_hub"

python - <<'PY'
import torch, transformers, trl, datasets, unsloth
print(f"torch={torch.__version__}  cuda={torch.version.cuda}")
print(f"transformers={transformers.__version__}  trl={trl.__version__}")
print(f"datasets={datasets.__version__}  unsloth={unsloth.__version__}")
print("✓ all deps OK")
PY


In [ ]:
%%bash
# Always do a clean clone — avoids all stale-file merge conflicts
rm -rf blindspot-env
git clone --depth 1 https://github.com/vasarlalikhilavinash/blindspot-env
cd blindspot-env
mkdir -p plots

echo "=== Trace data ==="
wc -l data/sft_traces.jsonl
python - <<'PY'
import json
lines = open("data/sft_traces.jsonl").readlines()
rewards = [json.loads(l)["final_reward"] for l in lines]
mean = sum(rewards)/len(rewards)
print(f"Traces: {len(lines)}  |  mean reward: {mean:+.3f}  |  min: {min(rewards):+.3f}  max: {max(rewards):+.3f}")
PY

echo ""
echo "=== Required data files ==="
for f in data/user_summaries.json data/user_splits.json data/concept_catalog.json \
         data/concept_pool_per_user.json data/ground_truth_adoption.json \
         data/comprehension_scores.json data/sft_traces.jsonl; do
  test -f "$f" && echo "  + $f" || echo "  MISSING: $f"
done
echo "done"


In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, 'blindspot-env')
DATA = Path('blindspot-env/data')

traces = [json.loads(l) for l in open(DATA / 'sft_traces.jsonl')]
users  = json.load(open(DATA / 'user_summaries.json'))
splits = json.load(open(DATA / 'user_splits.json'))

print(f"✓ {len(traces)} SFT traces loaded")
print(f"  train users : {len(splits.get('train', []))}")
print(f"  test  users : {len(splits.get('test', []))}")
print(f"  sample trace keys: {list(traces[0].keys())}")
print(f"  sample reward: {traces[0]['final_reward']:+.3f}")
print(f"  messages in trace: {len(traces[0]['messages'])}")


---
## 2. Pre-training analysis

Inspect the data the env was built on. This is the same pre-computed
artifact set that powers the reward function during training.

In [ ]:
import json, statistics
from pathlib import Path

DATA = Path('blindspot-env/data')

users = json.load(open(DATA/'user_summaries.json'))
catalog = json.load(open(DATA/'concept_catalog.json'))
pool = json.load(open(DATA/'concept_pool_per_user.json'))
adoption = json.load(open(DATA/'ground_truth_adoption.json'))
comp = json.load(open(DATA/'comprehension_scores.json'))
paths = json.load(open(DATA/'reading_paths.json'))
nov = json.load(open(DATA/'novelty_flags.json'))

stats = {
    'n_users': len(users),
    'n_concepts_total': len(catalog),
    'n_concepts_in_pool': len({c for v in pool.values() for c in v}),
    'n_reading_paths': len(paths),
    'n_novel_flagged': sum(1 for v in nov.values() if v),
    'n_adoption_pairs': sum(len(v) for v in adoption.values()),
    'n_comprehension_pairs': sum(len(v) for v in comp.values()),
    'pool_size_per_user_mean': round(statistics.mean(len(v) for v in pool.values()), 1),
    'adoption_per_user_mean': round(statistics.mean(len(v) for v in adoption.values()), 1),
    'comprehension_lift_mean': round(statistics.mean(
        l for v in comp.values() for l in v.values()) or 0, 3) if any(v for v in comp.values()) else 0,
}
import pandas as pd
display(pd.DataFrame(list(stats.items()), columns=['stat', 'value']).style.hide(axis='index'))

In [ ]:
# Distribution plots: pool sizes, adoption per user, path lengths, comprehension lifts
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

ax = axes[0, 0]
sizes = [len(v) for v in pool.values()]
ax.hist(sizes, bins=10, color='#3377cc', edgecolor='white')
ax.set_title(f'Concept pool size per user (n={len(pool)} users)')
ax.set_xlabel('# concepts in pool'); ax.set_ylabel('# users')
ax.axvline(np.mean(sizes), color='red', ls='--', lw=1, label=f'mean={np.mean(sizes):.1f}')
ax.legend()

ax = axes[0, 1]
adopt_counts = [len(v) for v in adoption.values()]
ax.hist(adopt_counts, bins=range(0, max(adopt_counts)+2), color='#22aa66', edgecolor='white')
ax.set_title('Ground-truth adoptions per user')
ax.set_xlabel('# adopted concepts'); ax.set_ylabel('# users')
ax.axvline(np.mean(adopt_counts), color='red', ls='--', lw=1, label=f'mean={np.mean(adopt_counts):.1f}')
ax.legend()

ax = axes[1, 0]
plens = [len(p) for p in paths.values()]
ax.hist(plens, bins=range(1, max(plens)+2), color='#cc7733', edgecolor='white')
ax.set_title('Reading-path length distribution')
ax.set_xlabel('# papers in path'); ax.set_ylabel('# concepts')

ax = axes[1, 1]
lifts = [l for v in comp.values() for l in v.values()]
if lifts:
    ax.hist(lifts, bins=8, color='#9966cc', edgecolor='white')
    ax.set_title(f'Comprehension lift distribution (n={len(lifts)} pairs)')
    ax.set_xlabel('judge accuracy lift'); ax.set_ylabel('# (user, concept) pairs')
    ax.axvline(np.mean(lifts), color='red', ls='--', lw=1, label=f'mean={np.mean(lifts):.3f}')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'no lift data', ha='center', va='center')
    ax.set_title('Comprehension lifts')

plt.tight_layout()
plt.savefig('blindspot-env/plots/data_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('blindspot-env/data')
traces = [json.loads(l) for l in open(DATA / 'sft_traces.jsonl')]
rewards = [t['final_reward'] for t in traces]

# Reward distribution of expert traces
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.hist(rewards, bins=12, color='#3377cc', edgecolor='white')
ax.axvline(np.mean(rewards), color='red', ls='--', lw=1.5, label=f'mean={np.mean(rewards):.2f}')
ax.set_title('Expert trace reward distribution')
ax.set_xlabel('episode reward'); ax.set_ylabel('# traces')
ax.legend()

ax = axes[1]
msg_lens = [len(t['messages']) for t in traces]
ax.hist(msg_lens, bins=10, color='#22aa66', edgecolor='white')
ax.axvline(np.mean(msg_lens), color='red', ls='--', lw=1.5, label=f'mean={np.mean(msg_lens):.1f}')
ax.set_title('Messages per expert trace')
ax.set_xlabel('# messages in trace'); ax.set_ylabel('# traces')
ax.legend()

plt.suptitle('SFT Expert Traces — Blindspot Dense Retrieval+', fontweight='bold')
plt.tight_layout()
plt.savefig('blindspot-env/plots/sft_trace_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Traces: {len(traces)}  |  mean reward: {np.mean(rewards):+.3f}  ± {np.std(rewards):.3f}")
print(f"Min: {min(rewards):+.3f}  Max: {max(rewards):+.3f}")


---
## 3. SFT Training

Load `Qwen2.5-1.5B-Instruct` in 4-bit, attach LoRA adapters (r=16),
and fine-tune for 1 epoch on the 40 expert traces using TRL's `SFTTrainer`.


In [ ]:
import os, torch
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

print(f"Loading {BASE_MODEL} in 4-bit ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto (bf16 on A100)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Loaded  |  trainable params: {n_params:,}")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  free {free/1e9:.1f}/{total/1e9:.1f} GB")


In [ ]:
from datasets import load_dataset

DATA_FILE = "blindspot-env/data/sft_traces.jsonl"
ds = load_dataset("json", data_files=DATA_FILE, split="train")
print(f"Dataset: {len(ds)} traces  |  columns: {ds.column_names}")

def format_trace(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )}

ds = ds.map(format_trace, remove_columns=ds.column_names)
print(f"✓ Formatted  |  example (first 300 chars):")
print(ds[0]["text"][:300])


In [ ]:
import torch, os, shutil
from pathlib import Path
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR  = "./blindspot-sft"
FINAL_DIR   = "./blindspot-sft-final"
PLOTS_DIR   = Path("blindspot-env/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    max_seq_length=2048,
    logging_steps=5,
    save_steps=50,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=sft_config,
)

FastLanguageModel.for_training(model)
print(f"Starting SFT: 1 epoch × {len(ds)} traces  "
      f"(effective batch = {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps})")
trainer.train()
print("✓ Training complete")

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✓ Adapter saved to {FINAL_DIR}")

# Copy trainer_state.json for plot cell below
state_src = Path(OUTPUT_DIR) / "trainer_state.json"
if state_src.exists():
    shutil.copy(state_src, PLOTS_DIR / "trainer_state.json")
    print("✓ trainer_state.json copied to plots/")


In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import ProcessorMixin

# CRITICAL: Unsloth returns a multimodal Qwen2_5_VLProcessor for Qwen3.5, not a plain tokenizer.
# When TRL calls processing_class(text=prompts_text, return_tensors="pt"), the processor tries
# to load images from the text and crashes with UnidentifiedImageError. Pass the inner text
# tokenizer instead so the entire vision codepath is bypassed.
text_tokenizer = tokenizer.tokenizer if isinstance(tokenizer, ProcessorMixin) else tokenizer
print(f"processing_class type: {type(text_tokenizer).__name__}  (was {type(tokenizer).__name__})")

cfg = GRPOConfig(
    output_dir='blindspot-env/training/checkpoints/grpo',
    learning_rate=5e-6,
    max_steps=120,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=2048,
    max_completion_length=96,
    logging_steps=5,
    save_steps=100,
    bf16=True,
    report_to='none',
)
trainer = GRPOTrainer(model=model, processing_class=text_tokenizer,
                      reward_funcs=[reward_fn], args=cfg, train_dataset=ds)

print(f'Starting GRPO: {cfg.max_steps} steps × {cfg.num_generations} rollouts/step '
      f'= {cfg.max_steps*cfg.num_generations} reward queries')
import os; os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
import torch; torch.cuda.empty_cache()
trainer.train()
trainer.save_model('blindspot-env/training/checkpoints/grpo')
print('✓ training complete, adapter saved')


---
## 4. Post-training analysis

Plot the SFT loss curve, then evaluate the trained policy against random
and trending baselines using the env server.


In [ ]:
# 4.1 SFT loss curve
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

state_path = Path("blindspot-env/plots/trainer_state.json")
state = json.loads(state_path.read_text())
history = state.get("log_history", [])
steps  = [e["step"] for e in history if "loss" in e]
losses = [e["loss"]  for e in history if "loss" in e]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, color='#3377cc', lw=2, marker='o', markersize=4)
ax.set_xlabel("Step"); ax.set_ylabel("Loss")
ax.set_title("SFT Training Loss — Blindspot Qwen2.5-1.5B")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('blindspot-env/plots/sft_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Steps: {len(steps)}  |  final loss: {losses[-1]:.4f}  |  initial loss: {losses[0]:.4f}")


In [ ]:
# 4.2 Boot env server for evaluation
import subprocess, time, requests

proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
    cwd="blindspot-env",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

ENV_URL = "http://localhost:8000"
for i in range(30):
    try:
        r = requests.post(f"{ENV_URL}/reset", json={}, timeout=3)
        if r.status_code == 200:
            obs = r.json().get("observation", r.json()) or {}
            print(f"✓ env server ready ({i+1}s)  |  users: {len(obs.get('user_id_pool', []))}  candidates: {len(obs.get('candidate_concepts', []))}")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("ERROR: server didn't start in 30s")


In [ ]:
# 4.3 Evaluate 3 policies: random, trending, SFT
import json, math, random, re, torch, requests, time
from pathlib import Path
from unsloth import FastLanguageModel

DATA   = Path("blindspot-env/data")
splits = json.load(open(DATA / "user_splits.json"))

r0 = requests.post(f"{ENV_URL}/reset", json={}).json()
user_pool   = (r0.get("observation", r0) or {}).get("user_id_pool", [])
train_users = [u for u in splits.get("train", []) if u in user_pool]
SEEDS = list(range(100, 103))  # 3 seeds per user → fast eval

_JSON_RE = re.compile(r"\{[^{}]*\}", re.DOTALL)

def _post(ep, payload):
    r = requests.post(f"{ENV_URL}/{ep}", json=payload, timeout=30)
    r.raise_for_status()
    return r.json()

def _get_reward(obs):
    rb = obs.get("reward_breakdown") or {}
    return float(rb.get("total", obs.get("reward", 0.0)))

def _parse(text):
    try:    return json.loads(text.strip())
    except: pass
    for m in _JSON_RE.finditer(text):
        try:    return json.loads(m.group(0))
        except: pass
    return None

# Random policy
def run_random(uid, seed):
    body  = _post("reset", {"user_id": uid, "seed": seed})
    obs   = body.get("observation", body)
    cands = obs.get("candidate_concepts", [])
    chosen = random.Random(seed).sample(cands, min(10, len(cands)))
    for c in chosen:
        res = _post("step", {"action": {"type": "surface", "concept_id": c["concept_id"]}})
        if (res.get("observation", res) or {}).get("done"): return _get_reward(res.get("observation", res))
    res = _post("step", {"action": {"type": "stop"}})
    return _get_reward(res.get("observation", res))

# Trending policy (first 10 in list order)
def run_trending(uid, seed):
    body  = _post("reset", {"user_id": uid, "seed": seed})
    obs   = body.get("observation", body)
    cands = obs.get("candidate_concepts", [])
    for c in cands[:10]:
        res = _post("step", {"action": {"type": "surface", "concept_id": c["concept_id"]}})
        if (res.get("observation", res) or {}).get("done"): return _get_reward(res.get("observation", res))
    res = _post("step", {"action": {"type": "stop"}})
    return _get_reward(res.get("observation", res))

# SFT policy
SYSTEM_PROMPT = (
    "You are Blindspot, a research discovery agent. "
    "Given a user profile and candidate concepts, choose actions to surface "
    "the most relevant unknown-unknown concepts the user has not seen yet. "
    'Respond with JSON: {"type": "inspect|surface|stop", "concept_id": int}'
)

def _render(obs):
    cands = obs.get("candidate_concepts", [])
    lines = [f"  id={c['concept_id']}: {c.get('title','')} — {c.get('one_liner','')[:80]}" for c in cands]
    return (f"USER PROFILE:\n{obs.get('user_summary','')[:800]}\n\n"
            f"BUDGETS: inspect={obs.get('inspect_budget_remaining')} "
            f"surface={obs.get('surface_budget_remaining')}\n\n"
            f"CANDIDATES:\n" + "\n".join(lines))

FastLanguageModel.for_inference(model)

def run_sft(uid, seed):
    body = _post("reset", {"user_id": uid, "seed": seed})
    obs  = body.get("observation", body)
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _render(obs)}]
    surfaced = set()
    for _ in range(12):
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp  = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        with torch.inference_mode():
            out = model.generate(inp, max_new_tokens=64, do_sample=False)
        reply  = tokenizer.decode(out[0, inp.shape[1]:], skip_special_tokens=True)
        action = _parse(reply) or {"type": "stop"}
        if action.get("type") == "surface":
            cid = action.get("concept_id")
            if cid in surfaced: action = {"type": "stop"}
            else: surfaced.add(cid)
        res     = _post("step", {"action": action})
        res_obs = res.get("observation", res)
        msgs.append({"role": "assistant", "content": reply})
        if res_obs.get("done"): return _get_reward(res_obs)
        msgs.append({"role": "user", "content": _render(res_obs)})
    res = _post("step", {"action": {"type": "stop"}})
    return _get_reward(res.get("observation", res))

# Run evaluation
results = {"random": [], "trending": [], "sft": []}
for uid in train_users:
    for s in SEEDS:
        results["random"].append(run_random(uid, s))
        results["trending"].append(run_trending(uid, s))
        results["sft"].append(run_sft(uid, s))
        print(f"  user={uid[:8]} seed={s}  rand={results['random'][-1]:+.2f}  "
              f"trend={results['trending'][-1]:+.2f}  sft={results['sft'][-1]:+.2f}")

import json
out = Path("blindspot-env/plots/eval_results.json")
out.write_text(json.dumps(results, indent=2))
print(f"\n✓ Saved to {out}")
for name, rws in results.items():
    m = sum(rws)/len(rws)
    s = math.sqrt(sum((r-m)**2 for r in rws)/len(rws))
    print(f"  {name:10s}: {m:+.3f} ± {s:.3f}  (n={len(rws)})")


In [ ]:
# 4.4 Bar chart: Random vs Trending vs SFT
import json, math, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

data = json.loads(Path("blindspot-env/plots/eval_results.json").read_text())
labels = list(data.keys())
means  = [sum(v)/len(v) for v in data.values()]
stds   = [math.sqrt(sum((r-m)**2 for r in v)/len(v)) for v, m in zip(data.values(), means)]
colors = ["#888888", "#cc7733", "#3377cc"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=7, color=colors,
              edgecolor="white", error_kw={"ecolor": "#333"})
ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.15,
            f"{m:+.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("Mean Episode Reward", fontsize=12)
ax.set_title("Blindspot — SFT Policy vs Baselines\n(mean ± std, train users)", fontsize=13)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("blindspot-env/plots/final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved blindspot-env/plots/final_comparison.png")


In [ ]:
# 4.5 Headline summary
import json, math
from pathlib import Path

data = json.loads(Path("blindspot-env/plots/eval_results.json").read_text())
print("=== RESULTS SUMMARY ===")
for name, rewards in data.items():
    m = sum(rewards)/len(rewards)
    s = math.sqrt(sum((r-m)**2 for r in rewards)/len(rewards))
    print(f"  {name:10s}: {m:+.3f} ± {s:.3f}  (n={len(rewards)})")

sft_mean   = sum(data["sft"])/len(data["sft"])
trend_mean = sum(data["trending"])/len(data["trending"])
rand_mean  = sum(data["random"])/len(data["random"])
print(f"\nSFT vs Trending : {sft_mean - trend_mean:+.3f}")
print(f"SFT vs Random   : {sft_mean - rand_mean:+.3f}")


---
## 5. Push adapter to Hugging Face Hub


In [ ]:
import os
# Set HF_TOKEN as a Colab secret (Secrets tab in left sidebar) or run:
#   import os; os.environ["HF_TOKEN"] = "hf_..."
HF_TOKEN = os.environ["HF_TOKEN"]   # will raise if not set — keeps token out of notebook
HUB_REPO  = "Vasarlaavinash/blindspot-sft-1.5b"

print(f"Pushing adapter to {HUB_REPO} ...")
model.push_to_hub(HUB_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_REPO, token=HF_TOKEN)
print(f"✓ Pushed to https://huggingface.co/{HUB_REPO}")
